In [ ]:
import numpy as np

# softplus(x) = log(1 + exp(x)).

softplus =  lambda x: np.log(1 + np.exp(x))
softplus_ = lambda x: (1 / (1 + np.exp(x))) * np.exp(x)


# set the layers - init
# forward pass
# store the z's and s's and p and y.
# max trick
# assert p = 1

# backward pass:
# softmax - CE.
# backprob grads, go through non-linearity, store weight grads

class FourLayerNet:
    def __init__(self, input_dim, h1_dim, h2_dim, h3_dim, num_classes):
      self.w1 = np.random.randn(input_dim, h1_dim) / np.sqrt(input_dim)
      self.w2 = np.random.randn(h1_dim, h2_dim) / np.sqrt(h1_dim)
      self.w3 = np.random.randn(h2_dim, h3_dim) / np.sqrt(h2_dim)
      self.w4 = np.random.randn(h3_dim, num_classes) / np.sqrt(h3_dim)


    def forward(self, X, y):
      _dict = {}
      _dict['x'] = X
      actives = X @ self.w1
      _dict['z1'] = actives
      actives = softplus(actives)
      _dict['a1'] = actives

      actives = actives @ self.w2
      _dict['z2'] = actives
      actives = (actives > 0 ) * actives
      _dict['a2'] = actives

      actives = actives @ self.w3
      _dict['z3'] = actives
      actives = (actives > 0 ) * actives
      _dict['a3'] = actives

      actives = actives @ self.w4
      #print(f"{actives.shape=}")


      actives = actives - actives.max(keepdims=True, axis=-1)
      actives = np.exp(actives)
      p = actives / actives.sum(axis=1, keepdims=True)
      #print(f"{p.sum(axis=1)}")
      assert np.allclose(p.sum(axis=1), 1.0)
      _dict['p'] = p
      _dict['y'] = y

      # loss
      b = X.shape[0]
      _dict['b'] = b

      self._dict = _dict

      _loss = -np.mean(np.log(p[np.arange(b), y]))
      return _loss

    def backward(self):
        grads = {}
        #print("*"*10)
        # CE + softmax
        y_hat = self._dict['p']
        y = self._dict['y']
        b = self._dict['b']

        y_ = np.zeros_like(y_hat)
        y_[np.arange(b), y] = 1

        grad = (y_hat - y_) / b # don't forget b
        # print("after CE")
        # print(abs(grad).mean())


        # ---------------
        grads['w4'] = self._dict['a3'].T @ grad
        grad = grad @ self.w4.T
        #grad = grad * softplus_(self._dict['z3'])
        grad = grad * (self._dict['z3'] > 0 ) #ReLu
        # print("after w4")
        # print(abs(grad).mean())

        # ---------------
        grads['w3'] = self._dict['a2'].T @ grad
        grad = grad @ self.w3.T
        grad = grad * (self._dict['z2'] > 0 ) #ReLu
        # print("after w3")
        # print(abs(grad).mean())
        # ---------------
        grads['w2'] = self._dict['a1'].T @ grad
        grad = grad @ self.w2.T
        grad = grad * softplus_(self._dict['z1'])
        # print("after w2")
        # print(abs(grad).mean())
        # ---------------
        grads['w1'] = self._dict['x'].T @ grad
        # print(abs(grad).mean())
        # print("*"*10)
        return grads



def test():
    net = FourLayerNet(10, 20, 15, 25, 3)
    X = np.random.randn(32, 10) * 0.5
    y = np.random.randint(0, 3, 32)

    for i in range(10000):
        loss = net.forward(X, y)
        #print(loss)
        grads = net.backward()
        # Update weights yourself
        lr_rate = 0.1
        net.w1 -= lr_rate * grads['w1']
        net.w2 -= lr_rate * grads['w2']
        net.w3 -= lr_rate * grads['w3']
        net.w4 -= lr_rate * grads['w4']
        if i % 20 == 0:
            print(f"Step {i}: Loss = {loss:.4f}")

test()

In [ ]:
softplus = lambda x: np.log(1 + np.exp(x))
softplus_ = lambda x: (1 / (1 + np.exp(x))) * np.exp(x)

In [ ]:
softplus(10)

np.float64(10.000045398899218)